In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
from scipy.stats import wilcoxon

### Defined the fold values (Random split)

In [ ]:
import pandas as pd
from pathlib import Path

RANDOM_SPLIT_DIR = Path("../results/train_output_summary/task1/random_split")

# Architecture files mapping
ARCH_FILES = {
    "SD-LogReg": "random_split_summary_logreg.csv",
    "SD-CNN": "random_split_summary_sdcnn.csv",
    "Evo2-CNN-zero-shot": "random_split_summary_evo2_zeroshot_token.csv",
    "Evo2-CNN-pca10": "random_split_summary_evo2_pca.csv",
    "MD-CNN": "random_split_summary_mdcnn.csv",
}

def load_fold_data_from_csv(filename):
    """Load 5-fold AUC values from CSV file, organized by drug."""
    df = pd.read_csv(RANDOM_SPLIT_DIR / filename)
    fold_data_dict = {}
    for drug in df['drug'].unique():
        drug_data = df[df['drug'] == drug].sort_values('fold')
        aucs = drug_data['val_auc'].tolist()
        fold_data_dict[drug.lower()] = aucs
    return fold_data_dict

# Load all fold data
fold_data = {}
for drug in ['capreomycin', 'amikacin', 'kanamycin', 'ethionamide', 'pyrazinamide', 
             'moxifloxacin', 'streptomycin', 'ethambutol', 'levofloxacin', 'isoniazid', 'rifampicin']:
    fold_data[drug] = {}
    for arch_name, filename in ARCH_FILES.items():
        arch_data = load_fold_data_from_csv(filename)
        if drug in arch_data:
            fold_data[drug][arch_name] = arch_data[drug]

print(f"[OK] Loaded fold data for {len(fold_data)} drugs and {len(ARCH_FILES)} architectures")

### Significance testing of SD cases with SD-Log Reg as the baseline across all drugs

In [3]:
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests


baseline = "SD-LogReg"
models = ["SD-CNN", "Evo2-CNN-zero-shot", "Evo2-CNN-pca10"]

results = []

for model in models:
    common_drugs = [
        d for d in fold_data
        if baseline in fold_data[d] and model in fold_data[d]
    ]

    baseline_means, model_means = [], []
    for drug in common_drugs:
        auc_base = np.array(fold_data[drug][baseline])
        auc_model = np.array(fold_data[drug][model])
        if len(auc_base) == len(auc_model) == 5:
            baseline_means.append(auc_base.mean())
            model_means.append(auc_model.mean())

    # paired Wilcoxon signed-rank test (since same drugs)
    w_stat, p_wilcoxon = wilcoxon(model_means, baseline_means, alternative='two-sided')

    results.append({
        "Model_vs_Baseline": model,
        "Mean_Baseline": np.mean(baseline_means),
        "Mean_Model": np.mean(model_means),
        "ΔAUC": np.mean(np.array(model_means) - np.array(baseline_means)),
        "W_stat": w_stat,
        "p_wilcoxon": p_wilcoxon,
        "Num_Drugs": len(common_drugs)
    })

# Convert to DataFrame
import pandas as pd
res_df = pd.DataFrame(results)
if not res_df.empty:
    res_df["p_wilcoxon_holm"] = multipletests(res_df["p_wilcoxon"], method="holm")[1]
print(res_df.to_string(index=False, float_format="%.6f"))

# Optionally save
res_df.to_csv("../results/significance_testing/sd_overall_model_vs_logreg_pvals.csv", index=False)
print("\n✅ Saved as SD_overall_model_vs_logreg_pvals.csv")

 Model_vs_Baseline  Mean_Baseline  Mean_Model      ΔAUC    W_stat  p_wilcoxon  Num_Drugs  p_wilcoxon_holm
            SD-CNN       0.867086    0.863593 -0.003493 32.000000    0.965820         11         1.000000
Evo2-CNN-zero-shot       0.867086    0.871453  0.004367 29.000000    0.764648         11         1.000000
    Evo2-CNN-pca10       0.867086    0.872846  0.005760 23.000000    0.413086         11         1.000000

✅ Saved as SD_overall_model_vs_logreg_pvals.csv


### Significance testing of SD cases with SD-CNN as the baseline across all drugs

In [4]:
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests


baseline = "SD-CNN"
models = ["Evo2-CNN-zero-shot", "Evo2-CNN-pca10"]

results = []

for model in models:
    common_drugs = [
        d for d in fold_data
        if baseline in fold_data[d] and model in fold_data[d]
    ]

    baseline_means, model_means = [], []
    for drug in common_drugs:
        auc_base = np.array(fold_data[drug][baseline])
        auc_model = np.array(fold_data[drug][model])
        if len(auc_base) == len(auc_model) == 5:
            baseline_means.append(auc_base.mean())
            model_means.append(auc_model.mean())

    # paired Wilcoxon signed-rank test (since same drugs)
    w_stat, p_wilcoxon = wilcoxon(model_means, baseline_means, alternative='two-sided')

    results.append({
        "Model_vs_Baseline": model,
        "Mean_Baseline": np.mean(baseline_means),
        "Mean_Model": np.mean(model_means),
        "ΔAUC": np.mean(np.array(model_means) - np.array(baseline_means)),
        "W_stat": w_stat,
        "p_wilcoxon": p_wilcoxon,
        "Num_Drugs": len(common_drugs)
    })

# Convert to DataFrame
import pandas as pd
res_df = pd.DataFrame(results)
if not res_df.empty:
    res_df["p_wilcoxon_holm"] = multipletests(res_df["p_wilcoxon"], method="holm")[1]
print(res_df.to_string(index=False, float_format="%.6f"))

# Optionally save
res_df.to_csv("../results/significance_testing/sd_overall_model_vs_sdcnn_pvals.csv", index=False)
print("\n✅ Saved as SD_overall_model_vs_sdcnn_pvals.csv")

 Model_vs_Baseline  Mean_Baseline  Mean_Model     ΔAUC    W_stat  p_wilcoxon  Num_Drugs  p_wilcoxon_holm
Evo2-CNN-zero-shot       0.863593    0.871453 0.007860 19.000000    0.240234         11         0.480469
    Evo2-CNN-pca10       0.863593    0.872846 0.009253 27.000000    0.637695         11         0.637695

✅ Saved as SD_overall_model_vs_sdcnn_pvals.csv


### Significance testing of MD cases with SD-LogReg as the baseline across all drugs

In [5]:
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests


baseline = "SD-LogReg"
models = ["MD-CNN"]

results = []

for model in models:
    common_drugs = [
        d for d in fold_data
        if baseline in fold_data[d] and model in fold_data[d]
    ]

    baseline_means, model_means = [], []
    for drug in common_drugs:
        auc_base = np.array(fold_data[drug][baseline])
        auc_model = np.array(fold_data[drug][model])
        if len(auc_base) == len(auc_model) == 5:
            baseline_means.append(auc_base.mean())
            model_means.append(auc_model.mean())

    # paired Wilcoxon signed-rank test (since same drugs)
    w_stat, p_wilcoxon = wilcoxon(model_means, baseline_means, alternative='two-sided')

    results.append({
        "Model_vs_Baseline": model,
        "Mean_Baseline": np.mean(baseline_means),
        "Mean_Model": np.mean(model_means),
        "ΔAUC": np.mean(np.array(model_means) - np.array(baseline_means)),
        "W_stat": w_stat,
        "p_wilcoxon": p_wilcoxon,
        "Num_Drugs": len(common_drugs)
    })

# Convert to DataFrame
import pandas as pd
res_df = pd.DataFrame(results)
if not res_df.empty:
    res_df["p_wilcoxon_holm"] = multipletests(res_df["p_wilcoxon"], method="holm")[1]
print(res_df.to_string(index=False, float_format="%.6f"))

# Optionally save
res_df.to_csv("../results/significance_testing/md_overall_model_vs_logreg_pvals.csv", index=False)
print("\n✅ Saved as MD_overall_model_vs_logreg_pvals.csv")

Model_vs_Baseline  Mean_Baseline  Mean_Model     ΔAUC   W_stat  p_wilcoxon  Num_Drugs  p_wilcoxon_holm
           MD-CNN       0.867086    0.922198 0.055112 0.000000    0.000977         11         0.000977

✅ Saved as MD_overall_model_vs_logreg_pvals.csv


### Significance testing of MD cases with SD-CNN as the baseline across all drugs

In [6]:
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests


baseline = "SD-CNN"
models = ["Evo2-CNN-zero-shot", "Evo2-CNN-pca10", "MD-CNN"]

results = []

for model in models:
    common_drugs = [
        d for d in fold_data
        if baseline in fold_data[d] and model in fold_data[d]
    ]

    baseline_means, model_means = [], []
    for drug in common_drugs:
        auc_base = np.array(fold_data[drug][baseline])
        auc_model = np.array(fold_data[drug][model])
        if len(auc_base) == len(auc_model) == 5:
            baseline_means.append(auc_base.mean())
            model_means.append(auc_model.mean())

    # paired Wilcoxon signed-rank test (since same drugs)
    w_stat, p_wilcoxon = wilcoxon(model_means, baseline_means, alternative='two-sided')

    results.append({
        "Model_vs_Baseline": model,
        "Mean_Baseline": np.mean(baseline_means),
        "Mean_Model": np.mean(model_means),
        "ΔAUC": np.mean(np.array(model_means) - np.array(baseline_means)),
        # "W_stat": w_stat,
        "p_wilcoxon": p_wilcoxon,
        "Num_Drugs": len(common_drugs)
    })

# Convert to DataFrame
import pandas as pd
res_df = pd.DataFrame(results)
if not res_df.empty:
    res_df["p_wilcoxon_holm"] = multipletests(res_df["p_wilcoxon"], method="holm")[1]
print(res_df.to_string(index=False, float_format="%.6f"))

# Optionally save
res_df.to_csv("../results/significance_testing/md_overall_model_vs_sdcnn_pvals.csv", index=False)
print("\n✅ Saved as overall_model_vs_sdcnn_pvals.csv")

 Model_vs_Baseline  Mean_Baseline  Mean_Model     ΔAUC  p_wilcoxon  Num_Drugs  p_wilcoxon_holm
Evo2-CNN-zero-shot       0.863593    0.871453 0.007860    0.240234         11         0.480469
    Evo2-CNN-pca10       0.863593    0.872846 0.009253    0.637695         11         0.637695
            MD-CNN       0.863593    0.922198 0.058605    0.000977         11         0.002930

✅ Saved as overall_model_vs_sdcnn_pvals.csv


### Significance testing of all cases with SD-LogReg as the baseline across all drugs

In [7]:
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests


baseline = "SD-LogReg"
models = ["SD-CNN", "Evo2-CNN-zero-shot", "Evo2-CNN-pca10", "MD-CNN"]

results = []

for model in models:
    common_drugs = [
        d for d in fold_data
        if baseline in fold_data[d] and model in fold_data[d]
    ]

    baseline_means, model_means = [], []
    for drug in common_drugs:
        auc_base = np.array(fold_data[drug][baseline])
        auc_model = np.array(fold_data[drug][model])
        if len(auc_base) == len(auc_model) == 5:
            baseline_means.append(auc_base.mean())
            model_means.append(auc_model.mean())

    # paired Wilcoxon signed-rank test (since same drugs)
    w_stat, p_wilcoxon = wilcoxon(model_means, baseline_means, alternative='two-sided')

    results.append({
        "Model_vs_Baseline": model,
        "Mean_Baseline": np.mean(baseline_means),
        "Mean_Model": np.mean(model_means),
        "ΔAUC": np.mean(np.array(model_means) - np.array(baseline_means)),
        # "W_stat": w_stat,
        "p_wilcoxon": p_wilcoxon,
        "Num_Drugs": len(common_drugs)
    })

# Convert to DataFrame
import pandas as pd
res_df = pd.DataFrame(results)
if not res_df.empty:
    res_df["p_wilcoxon_holm"] = multipletests(res_df["p_wilcoxon"], method="holm")[1]
print(res_df.to_string(index=False, float_format="%.6f"))

# Optionally save
res_df.to_csv("../results/significance_testing/overall_model_vs_logreg_pvals.csv", index=False)
print("\n✅ Saved as overall_model_vs_logreg_pvals.csv")

 Model_vs_Baseline  Mean_Baseline  Mean_Model      ΔAUC  p_wilcoxon  Num_Drugs  p_wilcoxon_holm
            SD-CNN       0.867086    0.863593 -0.003493    0.965820         11         1.000000
Evo2-CNN-zero-shot       0.867086    0.871453  0.004367    0.764648         11         1.000000
    Evo2-CNN-pca10       0.867086    0.872846  0.005760    0.413086         11         1.000000
            MD-CNN       0.867086    0.922198  0.055112    0.000977         11         0.003906

✅ Saved as overall_model_vs_logreg_pvals.csv


### Significance testing of all cases with SD-CNN as the baseline across all drugs

In [8]:
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests


baseline = "SD-CNN"
models = ["Evo2-CNN-zero-shot", "Evo2-CNN-pca10", "MD-CNN"]

results = []

for model in models:
    common_drugs = [
        d for d in fold_data
        if baseline in fold_data[d] and model in fold_data[d]
    ]

    baseline_means, model_means = [], []
    for drug in common_drugs:
        auc_base = np.array(fold_data[drug][baseline])
        auc_model = np.array(fold_data[drug][model])
        if len(auc_base) == len(auc_model) == 5:
            baseline_means.append(auc_base.mean())
            model_means.append(auc_model.mean())

    # paired Wilcoxon signed-rank test (since same drugs)
    w_stat, p_wilcoxon = wilcoxon(model_means, baseline_means, alternative='two-sided')

    results.append({
        "Model_vs_Baseline": model,
        "Mean_Baseline": np.mean(baseline_means),
        "Mean_Model": np.mean(model_means),
        "ΔAUC": np.mean(np.array(model_means) - np.array(baseline_means)),
        # "W_stat": w_stat,
        "p_wilcoxon": p_wilcoxon,
        "Num_Drugs": len(common_drugs)
    })

# Convert to DataFrame
import pandas as pd
res_df = pd.DataFrame(results)
if not res_df.empty:
    res_df["p_wilcoxon_holm"] = multipletests(res_df["p_wilcoxon"], method="holm")[1]
print(res_df.to_string(index=False, float_format="%.6f"))

# Optionally save
res_df.to_csv("../results/significance_testing/overall_model_vs_sdcnn_pvals.csv", index=False)
print("\n✅ Saved as overall_model_vs_sdcnn_pvals.csv")

 Model_vs_Baseline  Mean_Baseline  Mean_Model     ΔAUC  p_wilcoxon  Num_Drugs  p_wilcoxon_holm
Evo2-CNN-zero-shot       0.863593    0.871453 0.007860    0.240234         11         0.480469
    Evo2-CNN-pca10       0.863593    0.872846 0.009253    0.637695         11         0.637695
            MD-CNN       0.863593    0.922198 0.058605    0.000977         11         0.002930

✅ Saved as overall_model_vs_sdcnn_pvals.csv
